## Caminatas aleatorias I: Movimiento Browniano

## Teoría: Movimiento Browniano

El movimiento Browniano describe el movimiento aleatorio de partículas suspendidas en un fluido, resultado de las colisiones con las moléculas del medio. Fue observado por primera vez por Robert Brown en 1827 y explicado teóricamente por Albert Einstein en 1905.

Matemáticamente, el movimiento Browniano se modela como un proceso estocástico continuo, también conocido como caminata aleatoria continua o proceso de Wiener. La ecuación diferencial estocástica (EDS) que lo describe es:

$$
\frac{dx}{dt} = \sqrt{2D}\,\eta(t)
$$

donde $x(t)$ es la posición de la partícula, $D$ es el coeficiente de difusión y $\eta(t)$ es un ruido blanco gaussiano de media cero y correlación $\langle \eta(t)\eta(t') \rangle = \delta(t-t')$.

El resultado más importante es que la media cuadrática de la posición crece linealmente con el tiempo:

$$
\langle x^2(t) \rangle = 2Dt
$$

Este comportamiento es característico de la difusión normal.

In [ ]:
using Plots, Random, Distributions, Statistics, GLM, DataFrames #Librerías a usar

In [ ]:
function brownian_motion_generator(μ, D, Δt, steps, x0) #Función para generar una trayectoria de movimiento browniano
    traj = cumsum([x0; μ*Δt .+ sqrt(2D*Δt) .* rand(Normal(0,1),steps)])
    return traj
end

In [ ]:
function plot_trajectories(steps, Δt, trajectories) # Función para generar gráficas de múltiples trayectorias
    p = plot() # Create a new plot
    for traj in trajectories
        plot!(p, 0:Δt:Δt*steps, traj, label="", alpha=0.5)
    end
    xlabel!(p, "Time")
    ylabel!(p, "Position")
    title!(p, "Brownian Motion Trajectories")
    return p # Return the plot object
end

In [ ]:
function generate_trajectories(μ, D, Δt, steps, x0, num_trajectories) #Función para generar múltiples trayectorias de movimiento browniano
    trajectories = []
    for _ in 1:num_trajectories
        push!(trajectories, brownian_motion_generator(μ, D, Δt, steps, x0))
    end
    return trajectories
end

In [ ]:
μ, D, Δt, steps, x0 , number_of_trajectories = 0.0, 1.0, 0.1, 10000, 0.0, 100

trajectories = generate_trajectories(μ, D, Δt, steps, x0, number_of_trajectories)   #Generamos las trayectorias
plot_trajectories(steps, Δt, trajectories) #Generamos las gráficas de las trayectorias

In [ ]:
function compute_msd(trajectories) #Función para el cálculo del MSD    
    msd = mean(reduce(hcat, trajectories).^2, dims=2) |> vec #Une las trayectorias en una matriz y calcula el MSD promediando por cada columna, finalmente transforma en un vector
    return msd
end

In [ ]:
msd = compute_msd(trajectories, steps) #Calculamos el MSD

In [ ]:
time = 0:Δt:Δt*steps
#Creamos un DataFrame con los datos del MSD para regresión
df = DataFrame(time=time, msd=msd)
model = lm(@formula(msd ~ time), df)
msd_fit = predict(model)

# Creamos las gráficas
p = plot(time, msd, label="MSD", lw=2, color=:red, xlabel="Time", ylabel="MSD", title="MSD vs. Time with Linear Fit")
plot!(p, time, msd_fit, label="Linear fit", lw=2, color=:blue, linestyle=:dash)

# Guardamos la figura
#savefig(p, "msd_plot.png")

# Generamos la salida en pantalla
display(model)
p